In [25]:
import torch
from torchvision.models import (mobilenet_v3_small, MobileNet_V3_Small_Weights)

In [27]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

weights = MobileNet_V3_Small_Weights.DEFAULT

mobilenet = mobilenet_v3_small(weights=weights)
mobilenet.eval()
mobilenet.to(device)

preprocess = weights.transforms()

In [29]:
from PIL import Image

In [31]:
def extract_feature(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        feature = mobilenet.features(image_tensor)
        feature = mobilenet.avgpool(feature)
        feature = torch.flatten(feature, 1)

    return feature.squeeze(0).cpu()

In [33]:
feature = extract_feature("../data/labeled_data/img/C0002/after_1.png")

In [35]:
import torch
import torch.nn.functional as F

In [37]:
def build_event_feature(df):
    before_paths = [
        "../data/labeled_data" + df["before_1"],
        "../data/labeled_data" + df["before_2"],
        "../data/labeled_data" + df["before_3"]
    ]
    after_paths = [
        "../data/labeled_data" + df["after_1"],
        "../data/labeled_data" + df["after_2"],
        "../data/labeled_data" + df["after_3"]
    ]
    before_features = torch.stack([
        extract_feature(path) for path in before_paths
    ]).float()
    after_features = torch.stack([
        extract_feature(path) for path in after_paths
    ]).float()

    if torch.isnan(before_features).any():
        raise ValueError("before_features 中存在 NaN")
    if torch.isnan(after_features).any():
        raise ValueError("after_features 中存在 NaN")
    if torch.isinf(before_features).any():
        raise ValueError("before_features 中存在 Inf")
    if torch.isinf(after_features).any():
        raise ValueError("after_features 中存在 Inf")
        
    before_mean = before_features.mean(dim=0)
    after_mean = after_features.mean(dim=0)

    absolute_diff = torch.abs(before_mean - after_mean)

    similarities = []
    for before_feature in before_features:
        for after_feature in after_features:
            similarity = F.cosine_similarity(before_feature.unsqueeze(0),
                                            after_feature.unsqueeze(0), dim=1).item()
            similarities.append(similarity)

    if len(similarities) != 9:
        raise ValueError(f"相似度数量错误：{len(similarities)}")

    similarities_tensor = torch.tensor(
        similarities,
        dtype=torch.float32
    )

    if torch.isnan(similarities_tensor).any():
        raise ValueError("cosine similarities 中存在 NaN")

    median_similarity = torch.median(similarities_tensor).reshape(1)

    event_feature = torch.cat([absolute_diff, median_similarity])

    if torch.isnan(event_feature).any():
        raise ValueError("最终 event_feature 中存在 NaN")

    return event_feature

In [39]:
import numpy as np

In [41]:
def create_features(df):
    features = []
    valid_indices = []
    for index, row in df.iterrows():
        try: 
            feature = build_event_feature(row)
            features.append(feature.numpy())
            valid_indices.append(index)
        except Exception as error:
            print(f"跳过 video {row['ID']}: {error}")

    X = np.stack(features)
    valid_df = df.loc[valid_indices].copy()
    return X, valid_df

In [43]:
import pandas as pd

In [45]:
df_train = pd.read_excel("../data/labeled_data/train/train.xlsx")
df_test = pd.read_excel("../data/labeled_data/test/test.xlsx")

In [47]:
X_train, df_train_valid = create_features(df_train)

In [49]:
X_test, df_test_valid = create_features(df_test)

In [51]:
y_train = df_train_valid["consistency"].to_numpy()
y_test = df_test_valid["consistency"].to_numpy()

In [53]:
X_train.shape, X_test.shape

((280, 577), (71, 577))

In [55]:
from sklearn.linear_model import LogisticRegression

In [56]:
classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

classifier.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [59]:
probabilities = classifier.predict_proba(X_test)
predictions = classifier.predict(X_test)

In [61]:
import numpy as np

In [63]:
def final_decision(prob, classes, confidence_threshold=0.7, gap_threshold=0.2):
    prob = np.asarray(prob, dtype=float)
    classes = np.asarray(classes)
    ranked_indices = np.argsort(prob)[::-1]
    top_index = ranked_indices[0]
    top_prob = prob[top_index]
    second_prob = prob[ranked_indices[1]]
    prob_gap = top_prob - second_prob
    if top_prob < confidence_threshold:
        return 2
    if prob_gap < gap_threshold:
        return 2
    return int(classes[top_index])

In [234]:
confidence_threshold = 0.6
gap_threshold = 0.2
final_labels = [final_decision(prob, classes=classifier.classes_,confidence_threshold=confidence_threshold,gap_threshold=gap_threshold) for prob in probabilities]

In [236]:
results = df_test_valid[["ID"]].reset_index(drop=True).copy()
results["true_label"] = np.asarray(y_test, dtype=int)
results["prediction"] = final_labels

In [238]:
for column_index, class_label in enumerate(classifier.classes_):
    results[f"prob_{int(class_label)}"] = probabilities[:, column_index]

In [240]:
sorted_probabilities = np.sort(probabilities, axis=1)

In [242]:
results["top_probability"] = sorted_probabilities[:, -1]
results["probability_gap"] = (sorted_probabilities[:, -1] - sorted_probabilities[:, -2])

In [244]:
results["uncertain_reason"] = ["; ".join(
        reason
        for condition, reason in [
            (top < confidence_threshold, "最高概率不足"),
            (gap < gap_threshold, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results["top_probability"],
        results["probability_gap"]
    )
]

In [246]:
correct_count = (results["true_label"] == results["prediction"]).sum()
uncertain_count = (results["prediction"] == 2).sum()

In [248]:
print("分类器类别：", classifier.classes_)
print("当前参与预测的测试样本数：", len(results))
print("预测正确数：", correct_count)
print(f"包含拒判的标签准确率：{correct_count / len(results):.2%}")
print("Uncertain 数量：", uncertain_count)
print("\n预测类别数量：")
print(results["prediction"].value_counts().sort_index())

display(results)

分类器类别： [0 1]
当前参与预测的测试样本数： 71
预测正确数： 53
包含拒判的标签准确率：74.65%
Uncertain 数量： 8

预测类别数量：
prediction
0    57
1     6
2     8
Name: count, dtype: int64


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,uncertain_reason
0,C0495,0,1,0.350446,0.649554,0.649554,0.299107,
1,C0356,0,0,0.695646,0.304354,0.695646,0.391293,
2,C0471,0,0,0.964776,0.035224,0.964776,0.929552,
3,C0115,0,0,0.886529,0.113471,0.886529,0.773058,
4,C0482,0,0,0.967710,0.032290,0.967710,0.935419,
...,...,...,...,...,...,...,...,...
66,C0063,0,0,0.930541,0.069459,0.930541,0.861082,
67,C0465,0,0,0.900661,0.099339,0.900661,0.801321,
68,C0367,0,0,0.779251,0.220749,0.779251,0.558501,
69,C0216,0,0,0.781741,0.218259,0.781741,0.563482,


In [250]:
uncertain_count

np.int64(8)

In [252]:
correct_count

np.int64(53)

In [254]:
table = pd.crosstab(
    results["true_label"],
    results["prediction"]
).reindex(
    index=[0, 1],
    columns=[0, 1, 2],
    fill_value=0
)

table.index = ["真实 Pass", "真实 Fail"]
table.columns = ["预测 Pass", "预测 Fail", "Uncertain"]

display(table)

,预测 Pass,预测 Fail,Uncertain
真实 Pass,51,4,6
真实 Fail,6,2,2


In [256]:
display(
    results.loc[
        results["true_label"] == 1,
        [
            "ID",
            "true_label",
            "prediction",
            "prob_0",
            "prob_1",
            "top_probability",
            "probability_gap",
            "uncertain_reason"
        ]
    ].sort_values(["prediction", "prob_1"])
)

,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,uncertain_reason
44,C0274,1,0,0.986580,0.013420,0.986580,0.973159,
12,C0301,1,0,0.910103,0.089897,0.910103,0.820205,
52,C0181,1,0,0.904466,0.095534,0.904466,0.808932,
43,C0272,1,0,0.872326,0.127674,0.872326,0.744653,
35,C0273,1,0,0.788026,0.211974,0.788026,0.576052,
61,C0361,1,0,0.688034,0.311966,0.688034,0.376068,
56,C0087,1,1,0.250258,0.749742,0.749742,0.499483,
53,C0493,1,1,0.241500,0.758500,0.758500,0.516999,
45,C0020,1,2,0.555883,0.444117,0.555883,0.111767,最高概率不足; 前两类概率接近
57,C0385,1,2,0.465063,0.534937,0.534937,0.069874,最高概率不足; 前两类概率接近


In [269]:
fail_ids = results.loc[results["true_label"] == 1, "ID"]

df_test_matched = df_test[
    df_test["ID"].isin(fail_ids)
].copy()

display(df_test_matched)

,ID,consistency,reason,color,shape,occlusion,reappearance_time,before_1,before_2,before_3,after_1,after_2,after_3,清晰度评分（1–5）,流畅度评分（1–5）,before_color,after_color,before_shape,after_shape,Unnamed: 9
12,C0301,1,小窗及出口出现强烈光晕，重现球体变为橙金色发光外观，与遮挡前明显不同。,灰粉色→橙金色发光,球形→球形,2.6,3.1,/img/C0301/before_1.png,/img/C0301/before_2.png,/img/C0301/before_3.png,/img/C0301/after_1.png,/img/C0301/after_2.png,/img/C0301/after_3.png,4,4,50,31,76,76,NaN
35,C0273,1,左侧金色球经隧道后在右侧呈明显橙红色，遮挡前后颜色不一致。,金色、棕色→橙红色、浅棕色,两个球体→两个球体,2.5,3.7,/img/C0273/before_1.png,/img/C0273/before_2.png,/img/C0273/before_3.png,/img/C0273/after_1.png,/img/C0273/after_2.png,/img/C0273/after_3.png,4,3,83,29,18,18,NaN
43,C0272,1,后段左侧同时出现两个球，右侧另有一球，目标数量增加。,金色、棕色→金色、棕色,两个球体→至少三个球体,1.6,3.1,/img/C0272/before_1.png,/img/C0272/before_2.png,/img/C0272/before_3.png,/img/C0272/after_1.png,/img/C0272/after_2.png,/img/C0272/after_3.png,3,3,83,83,18,79,NaN
44,C0274,1,左侧目标重现后由浅棕转为红棕，末段又出现青色块，颜色发生异常变化。,浅棕色、红棕色→红棕色、青红双色,两个球体→两个球体,0.7,2.0,/img/C0274/before_1.png,/img/C0274/before_2.png,/img/C0274/before_3.png,/img/C0274/after_1.png,/img/C0274/after_2.png,/img/C0274/after_3.png,4,3,33,74,18,18,NaN
45,C0020,1,重现后约3.1至3.4秒画面同时出现两个绿色球，数量由一个变为两个。,绿色→绿色,小球体→两个小球体（后恢复一个）,1.8,2.8,/img/C0020/before_1.png,/img/C0020/before_2.png,/img/C0020/before_3.png,/img/C0020/after_1.png,/img/C0020/after_2.png,/img/C0020/after_3.png,3,3,3,3,42,14,NaN
52,C0181,1,补帧确认约2.35秒左侧球体尚未完全遮住，右侧已出现另一球体，存在重复与运动衔接异常。,灰绿色→灰绿色,单个球体→两侧同时出现球体,2.3,2.3,/img/C0181/before_1.png,/img/C0181/before_2.png,/img/C0181/before_3.png,/img/C0181/after_1.png,/img/C0181/after_2.png,/img/C0181/after_3.png,5,3,52,52,33,20,NaN
53,C0493,1,原青蓝球体被遮住后，约2.8秒左侧新增灰白球体；随后青蓝球体在右侧重现，目标数量及出现过程异常。,青蓝色→青蓝色及灰白色,单个球体→新增灰白球体,1.2,2.8,/img/C0493/before_1.png,/img/C0493/before_2.png,/img/C0493/before_3.png,/img/C0493/after_1.png,/img/C0493/after_2.png,/img/C0493/after_3.png,5,3,92,98,33,58,NaN
56,C0087,1,前几次重现基本一致，后段约4.5秒后摆锤出现明显扁化、残缺。,棕金色→棕金色,球形摆锤→扁化不规则摆锤,0.4,0.5,/img/C0087/before_1.png,/img/C0087/before_2.png,/img/C0087/before_3.png,/img/C0087/after_1.png,/img/C0087/after_2.png,/img/C0087/after_3.png,4,3,25,25,78,54,NaN
57,C0385,1,后段目标穿过模糊扭曲的轨道后，尺寸相对柱体和轨道明显增大，前后不一致。,青绿色→青绿色,小球形→明显变大的球形,1.5,1.9,/img/C0385/before_1.png,/img/C0385/before_2.png,/img/C0385/before_3.png,/img/C0385/after_1.png,/img/C0385/after_2.png,/img/C0385/after_3.png,3,2,86,86,44,68,NaN
61,C0361,1,原球仍在上层时中央框内出现另一球，后段同时可见两球，数量不一致。,紫红色→紫红色,一个球体→两个球体,0.8,0.8,/img/C0361/before_1.png,/img/C0361/before_2.png,/img/C0361/before_3.png,/img/C0361/after_1.png,/img/C0361/after_2.png,/img/C0361/after_3.png,5,4,63,63,7,18,NaN


# cross_validation

In [271]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# 沿用前面已经提取好的特征与对应样本
cv_y_train = np.asarray(y_train, dtype=int)
cv_y_test = np.asarray(y_test, dtype=int)

assert len(X_train) == len(df_train_valid) == len(cv_y_train)
assert len(X_test) == len(df_test_valid) == len(cv_y_test)
assert set(np.unique(cv_y_train)) == {0, 1}

assert not (
    set(df_train_valid["ID"]) & set(df_test_valid["ID"])
), "训练集和测试集存在重复 ID"

# 如果有真实的关联分组列，填写列名，例如 "task_group"
# 不要使用每条视频独有的 ID 代替关联分组
CV_GROUP_COLUMN = None

if CV_GROUP_COLUMN is None:
    if pd.Series(cv_y_train).value_counts().min() < 5:
        raise ValueError("某个类别不足 5 条，无法进行分层五折")

    cv_splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_folds = list(
        cv_splitter.split(X_train, cv_y_train)
    )

    print("使用探索性分层五折：尚未控制关联视频分组")

else:
    cv_groups = df_train_valid[CV_GROUP_COLUMN]

    if cv_groups.isna().any():
        raise ValueError("存在缺失的分组信息")

    cv_groups = cv_groups.to_numpy()

    cv_splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_folds = list(
        cv_splitter.split(X_train, cv_y_train, cv_groups)
    )

    print("使用分层分组五折交叉验证")


# 每条训练样本获得一次折外预测
# 概率列固定为 0=Pass，1=Fail
oof_probabilities = np.full((len(cv_y_train), 2), np.nan)
oof_fold = np.full(len(cv_y_train), -1)

for fold, (fit_index, valid_index) in enumerate(cv_folds, start=1):

    if set(np.unique(cv_y_train[fit_index])) != {0, 1}:
        raise ValueError(f"第 {fold} 折训练部分缺少一个类别")

    if CV_GROUP_COLUMN is not None:
        assert not (
            set(cv_groups[fit_index]) & set(cv_groups[valid_index])
        ), "同一分组同时进入了训练折和验证折"

    # 复制原分类器的设置，但不复制其已训练参数
    fold_classifier = clone(classifier)

    fold_classifier.fit(
        X_train[fit_index],
        cv_y_train[fit_index]
    )

    fold_probabilities = fold_classifier.predict_proba(
        X_train[valid_index]
    )

    for column, label in enumerate(fold_classifier.classes_):
        oof_probabilities[valid_index, int(label)] = (
            fold_probabilities[:, column]
        )

    oof_fold[valid_index] = fold

    print(
        f"第 {fold} 折完成："
        f"训练 {len(fit_index)} 条，"
        f"验证 {len(valid_index)} 条"
    )

assert np.isfinite(oof_probabilities).all()
assert (oof_fold > 0).all()

oof_results = pd.DataFrame({
    "ID": df_train_valid["ID"].to_numpy(),
    "fold": oof_fold,
    "true_label": cv_y_train,
    "prob_0": oof_probabilities[:, 0],
    "prob_1": oof_probabilities[:, 1]
})

display(oof_results.head())

使用探索性分层五折：尚未控制关联视频分组
第 1 折完成：训练 224 条，验证 56 条
第 2 折完成：训练 224 条，验证 56 条
第 3 折完成：训练 224 条，验证 56 条
第 4 折完成：训练 224 条，验证 56 条
第 5 折完成：训练 224 条，验证 56 条


,ID,fold,true_label,prob_0,prob_1
0,C0033,1,0,0.818720,0.181280
1,C0066,1,0,0.873006,0.126994
2,C0192,4,0,0.728790,0.271210
3,C0324,3,1,0.500085,0.499915
4,C0255,5,0,0.708701,0.291299


In [273]:
CV_GAP_THRESHOLD = 0.2
CV_MIN_COVERAGE = 0.70

cv_candidates = np.round(
    np.arange(0.50, 0.951, 0.01), 2
)


def cv_predict_with_threshold(probabilities, threshold):
    return np.array([
        final_decision(
            prob,
            classes=np.array([0, 1]),
            confidence_threshold=threshold,
            gap_threshold=CV_GAP_THRESHOLD
        )
        for prob in probabilities
    ], dtype=int)


def cv_evaluate(y_true, predictions):
    decided = predictions != 2
    fail_mask = y_true == 1

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        predictions,
        labels=[0, 1],
        zero_division=0
    )

    decided_fail_count = (decided & fail_mask).sum()
    correct_fail_count = ((predictions == 1) & fail_mask).sum()

    return {
        "accuracy_all": accuracy_score(y_true, predictions),
        "accuracy_decided": (
            accuracy_score(y_true[decided], predictions[decided])
            if decided.any() else np.nan
        ),
        "coverage": decided.mean(),
        "abstention_rate": (~decided).mean(),
        "macro_f1": f1.mean(),
        "pass_recall": recall[0],
        "fail_recall_all": recall[1],
        "fail_precision": precision[1],
        "fail_coverage": (
            decided[fail_mask].mean()
            if fail_mask.any() else np.nan
        ),
        "fail_recall_decided": (
            correct_fail_count / decided_fail_count
            if decided_fail_count > 0 else np.nan
        ),
        "uncertain_count": int((~decided).sum())
    }


cv_rows = []

for threshold in cv_candidates:
    cv_predictions = cv_predict_with_threshold(
        oof_probabilities,
        threshold
    )

    cv_rows.append({
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_train, cv_predictions)
    })

cv_threshold_table = pd.DataFrame(cv_rows)

cv_eligible = cv_threshold_table.loc[
    cv_threshold_table["coverage"] >= CV_MIN_COVERAGE
].copy()

if cv_eligible.empty:
    raise ValueError("没有候选阈值满足预设的 70% 覆盖率要求")

cv_ranked = cv_eligible.sort_values(
    by=[
        "macro_f1",
        "fail_recall_all",
        "coverage",
        "confidence_threshold"
    ],
    ascending=[False, False, False, True]
)

cv_best_threshold = float(
    cv_ranked.iloc[0]["confidence_threshold"]
)

print("选出的 confidence_threshold：", cv_best_threshold)
print("固定 gap_threshold：", CV_GAP_THRESHOLD)

display(cv_ranked.head(10).round(4))

选出的 confidence_threshold： 0.5
固定 gap_threshold： 0.2


,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
0,0.50,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
1,0.51,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
2,0.52,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
3,0.53,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
4,0.54,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
5,0.55,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
6,0.56,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
7,0.57,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
8,0.58,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39
9,0.59,0.7393,0.8589,0.8607,0.1393,0.5103,0.8354,0.1081,0.2857,0.7568,0.1429,39


In [277]:
cv_final_classifier = clone(classifier)

cv_final_classifier.fit(X_train, cv_y_train)

cv_raw_test_probabilities = cv_final_classifier.predict_proba(X_test)

# 概率列统一为 0=Pass、1=Fail
cv_test_probabilities = np.zeros((len(cv_y_test), 2))

for column, label in enumerate(cv_final_classifier.classes_):
    cv_test_probabilities[:, int(label)] = (
        cv_raw_test_probabilities[:, column]
    )

cv_test_predictions = cv_predict_with_threshold(
    cv_test_probabilities,
    cv_best_threshold
)

results_cv = pd.DataFrame({
    "ID": df_test_valid["ID"].to_numpy(),
    "true_label": cv_y_test,
    "prediction": cv_test_predictions,
    "prob_0": cv_test_probabilities[:, 0],
    "prob_1": cv_test_probabilities[:, 1],
    "top_probability": cv_test_probabilities.max(axis=1),
    "probability_gap": np.abs(
        cv_test_probabilities[:, 0] - cv_test_probabilities[:, 1]
    )
})

results_cv["uncertain_reason"] = [
    "; ".join(
        reason
        for condition, reason in [
            (top < cv_best_threshold, "最高概率不足"),
            (gap < CV_GAP_THRESHOLD, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results_cv["top_probability"],
        results_cv["probability_gap"]
    )
]

table_cv = pd.crosstab(
    results_cv["true_label"],
    results_cv["prediction"]
).reindex(
    index=[0, 1],
    columns=[0, 1, 2],
    fill_value=0
)

table_cv.index = ["真实 Pass", "真实 Fail"]
table_cv.columns = ["预测 Pass", "预测 Fail", "Uncertain"]

print("交叉验证阈值对应的测试集混淆矩阵")
display(table_cv)

cv_comparison_rows = []

# for name, threshold in [
#     ("固定阈值 0.70", 0.70),
#     ("交叉验证选择阈值", cv_best_threshold)
# ]:
#     predictions_for_comparison = cv_predict_with_threshold(
#         cv_test_probabilities,
#         threshold
#     )

#     cv_comparison_rows.append({
#         "method": name,
#         "confidence_threshold": threshold,
#         **cv_evaluate(cv_y_test, predictions_for_comparison)
#     })

cv_test_comparison = pd.DataFrame(cv_comparison_rows)

display(cv_test_comparison.round(4))

print("真实 Fail 的预测明细")
display(
    results_cv.loc[
        results_cv["true_label"] == 1
    ].sort_values(["prediction", "prob_1"])
)

交叉验证阈值对应的测试集混淆矩阵


,预测 Pass,预测 Fail,Uncertain
真实 Pass,51,4,6
真实 Fail,6,2,2


""


真实 Fail 的预测明细


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,uncertain_reason
44,C0274,1,0,0.986580,0.013420,0.986580,0.973159,
12,C0301,1,0,0.910103,0.089897,0.910103,0.820205,
52,C0181,1,0,0.904466,0.095534,0.904466,0.808932,
43,C0272,1,0,0.872326,0.127674,0.872326,0.744653,
35,C0273,1,0,0.788026,0.211974,0.788026,0.576052,
61,C0361,1,0,0.688034,0.311966,0.688034,0.376068,
56,C0087,1,1,0.250258,0.749742,0.749742,0.499483,
53,C0493,1,1,0.241500,0.758500,0.758500,0.516999,
45,C0020,1,2,0.555883,0.444117,0.555883,0.111767,前两类概率接近
57,C0385,1,2,0.465063,0.534937,0.534937,0.069874,前两类概率接近


# Color

In [280]:
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# 检查前面生成的变量
assert len(X_train) == len(df_train_valid)
assert len(X_test) == len(df_test_valid)

assert np.array_equal(
    cv_y_train,
    df_train_valid["consistency"].to_numpy(dtype=int)
)

assert np.array_equal(
    cv_y_test,
    df_test_valid["consistency"].to_numpy(dtype=int)
)

COLOR_COLUMNS = ["before_color", "after_color"]


def get_color_codes(df):
    # 必须使用 valid 表，才能与 X_train / X_test 行顺序一致
    codes = df[COLOR_COLUMNS].apply(
        pd.to_numeric,
        errors="raise"
    ).to_numpy(dtype=float)

    if not np.isfinite(codes).all():
        raise ValueError("颜色编号存在空值或无穷值，请先检查")

    if not np.equal(codes, np.floor(codes)).all():
        raise ValueError("颜色编号应为整数")

    return codes


train_color_codes = get_color_codes(df_train_valid)
test_color_codes = get_color_codes(df_test_valid)

# 原始 MobileNet 事件特征维度
appearance_dimension = X_train.shape[1]

# 把原始颜色编号附在图像特征后面
# 编号稍后在 Pipeline 内按类别编码
X_train_with_color = np.column_stack([
    X_train,
    train_color_codes
])

X_test_with_color = np.column_stack([
    X_test,
    test_color_codes
])


def color_change_features(color_codes):
    before = color_codes[:, 0]
    after = color_codes[:, 1]

    # 只有两个编号都为正数，才认为颜色信息有效
    valid_pair = (before > 0) & (after > 0)

    changed = valid_pair & (before != after)

    # 不把未知、未再次出现等负数状态直接视为颜色变化
    return np.column_stack([
        valid_pair.astype(float),
        changed.astype(float)
    ])


def make_color_appearance_model():
    try:
        color_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        # 兼容较早版本的 sklearn
        color_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    transformer = ColumnTransformer(
        transformers=[
            (
                "appearance",
                "passthrough",
                list(range(appearance_dimension))
            ),
            (
                "color_categories",
                color_encoder,
                [
                    appearance_dimension,
                    appearance_dimension + 1
                ]
            ),
            (
                "color_change",
                FunctionTransformer(
                    color_change_features,
                    validate=False
                ),
                [
                    appearance_dimension,
                    appearance_dimension + 1
                ]
            )
        ],
        sparse_threshold=0
    )

    return Pipeline([
        ("features", transformer),

        # 沿用原逻辑回归的超参数
        ("classifier", clone(classifier))
    ])


print("原图像特征维度：", appearance_dimension)
print("训练样本数：", len(X_train_with_color))
print("测试样本数：", len(X_test_with_color))

原图像特征维度： 577
训练样本数： 280
测试样本数： 71


In [282]:
color_oof_probabilities = np.full(
    (len(cv_y_train), 2),
    np.nan
)

for fold, (fit_index, valid_index) in enumerate(cv_folds, start=1):

    color_fold_model = make_color_appearance_model()

    # One-Hot 编码器和逻辑回归都只拟合这一折的训练部分
    color_fold_model.fit(
        X_train_with_color[fit_index],
        cv_y_train[fit_index]
    )

    fold_probabilities = color_fold_model.predict_proba(
        X_train_with_color[valid_index]
    )

    fold_classes = color_fold_model.named_steps[
        "classifier"
    ].classes_

    for column, label in enumerate(fold_classes):
        color_oof_probabilities[valid_index, int(label)] = (
            fold_probabilities[:, column]
        )

    print(f"颜色联合模型：第 {fold} 折完成")

assert np.isfinite(color_oof_probabilities).all()

color_cv_rows = []

for threshold in cv_candidates:
    color_predictions = cv_predict_with_threshold(
        color_oof_probabilities,
        threshold
    )

    color_cv_rows.append({
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_train, color_predictions)
    })

color_cv_threshold_table = pd.DataFrame(color_cv_rows)

color_cv_eligible = color_cv_threshold_table.loc[
    color_cv_threshold_table["coverage"] >= CV_MIN_COVERAGE
].copy()

if color_cv_eligible.empty:
    raise ValueError("颜色联合模型没有阈值满足预设覆盖率要求")

color_cv_ranked = color_cv_eligible.sort_values(
    by=[
        "macro_f1",
        "fail_recall_all",
        "coverage",
        "confidence_threshold"
    ],
    ascending=[False, False, False, True]
)

color_best_threshold = float(
    color_cv_ranked.iloc[0]["confidence_threshold"]
)

print("颜色联合模型选择的置信度阈值：", color_best_threshold)
print("固定概率差阈值：", CV_GAP_THRESHOLD)

display(color_cv_ranked.head(10).round(4))

颜色联合模型：第 1 折完成
颜色联合模型：第 2 折完成
颜色联合模型：第 3 折完成
颜色联合模型：第 4 折完成
颜色联合模型：第 5 折完成
颜色联合模型选择的置信度阈值： 0.61
固定概率差阈值： 0.2


,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
11,0.61,0.7857,0.880,0.8929,0.1071,0.6260,0.8601,0.2973,0.4583,0.7568,0.3929,30
0,0.50,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
1,0.51,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
2,0.52,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
3,0.53,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
4,0.54,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
5,0.55,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
6,0.56,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
7,0.57,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26
8,0.58,0.7929,0.874,0.9071,0.0929,0.6245,0.8683,0.2973,0.4400,0.7838,0.3793,26


In [284]:
color_final_model = make_color_appearance_model()

color_final_model.fit(
    X_train_with_color,
    cv_y_train
)

color_raw_test_probabilities = color_final_model.predict_proba(
    X_test_with_color
)

color_test_probabilities = np.zeros((len(cv_y_test), 2))

color_classes = color_final_model.named_steps[
    "classifier"
].classes_

for column, label in enumerate(color_classes):
    color_test_probabilities[:, int(label)] = (
        color_raw_test_probabilities[:, column]
    )

color_test_predictions = cv_predict_with_threshold(
    color_test_probabilities,
    color_best_threshold
)

results_color = pd.DataFrame({
    "ID": df_test_valid["ID"].to_numpy(),
    "true_label": cv_y_test,
    "prediction": color_test_predictions,
    "prob_0": color_test_probabilities[:, 0],
    "prob_1": color_test_probabilities[:, 1],
    "top_probability": color_test_probabilities.max(axis=1),
    "probability_gap": np.abs(
        color_test_probabilities[:, 0]
        - color_test_probabilities[:, 1]
    ),
    "before_color": train_color_codes[:0, 0]
        if len(cv_y_test) == 0 else test_color_codes[:, 0],
    "after_color": test_color_codes[:, 1]
})

results_color["uncertain_reason"] = [
    "; ".join(
        reason
        for condition, reason in [
            (top < color_best_threshold, "最高概率不足"),
            (gap < CV_GAP_THRESHOLD, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results_color["top_probability"],
        results_color["probability_gap"]
    )
]

table_color = pd.crosstab(
    results_color["true_label"],
    results_color["prediction"]
).reindex(
    index=[0, 1],
    columns=[0, 1, 2],
    fill_value=0
)

table_color.index = ["真实 Pass", "真实 Fail"]
table_color.columns = ["预测 Pass", "预测 Fail", "Uncertain"]

print("MobileNet＋颜色联合模型的测试混淆矩阵")
display(table_color)

# 同一测试集，比较原模型与颜色联合模型
color_comparison_rows = []

for name, probs, threshold in [
    (
        "原模型：固定阈值 0.70",
        cv_test_probabilities,
        0.70
    ),
    (
        "原模型：交叉验证阈值",
        cv_test_probabilities,
        cv_best_threshold
    ),
    (
        "MobileNet＋颜色：交叉验证阈值",
        color_test_probabilities,
        color_best_threshold
    )
]:
    predictions_for_comparison = cv_predict_with_threshold(
        probs,
        threshold
    )

    color_comparison_rows.append({
        "method": name,
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_test, predictions_for_comparison)
    })

color_test_comparison = pd.DataFrame(color_comparison_rows)

display(color_test_comparison.round(4))

print("真实 Fail 的预测明细")
display(
    results_color.loc[
        results_color["true_label"] == 1
    ].sort_values(["prediction", "prob_1"])
)

MobileNet＋颜色联合模型的测试混淆矩阵


,预测 Pass,预测 Fail,Uncertain
真实 Pass,52,4,5
真实 Fail,6,4,0


,method,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
0,原模型：固定阈值 0.70,0.70,0.6338,0.8491,0.7465,0.2535,0.5278,0.7049,0.2,0.4000,0.7,0.2857,18
1,原模型：交叉验证阈值,0.50,0.7465,0.8413,0.8873,0.1127,0.5572,0.8361,0.2,0.3333,0.8,0.2500,8
2,MobileNet＋颜色：交叉验证阈值,0.61,0.7887,0.8485,0.9296,0.0704,0.6592,0.8525,0.4,0.5000,1.0,0.4000,5


真实 Fail 的预测明细


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,before_color,after_color,uncertain_reason
52,C0181,1,0,0.955893,0.044107,0.955893,0.911786,52.0,52.0,
45,C0020,1,0,0.931394,0.068606,0.931394,0.862788,3.0,3.0,
44,C0274,1,0,0.914032,0.085968,0.914032,0.828064,33.0,74.0,
43,C0272,1,0,0.881297,0.118703,0.881297,0.762595,83.0,83.0,
61,C0361,1,0,0.832880,0.167120,0.832880,0.665759,63.0,63.0,
12,C0301,1,0,0.797006,0.202994,0.797006,0.594011,50.0,31.0,
35,C0273,1,1,0.291018,0.708982,0.708982,0.417965,83.0,29.0,
56,C0087,1,1,0.248443,0.751557,0.751557,0.503113,25.0,25.0,
57,C0385,1,1,0.210425,0.789575,0.789575,0.579150,86.0,86.0,
53,C0493,1,1,0.160889,0.839111,0.839111,0.678222,92.0,98.0,


# shape

In [287]:
SHAPE_COLUMNS = ["before_shape", "after_shape"]


def get_shape_codes(df):
    codes = df[SHAPE_COLUMNS].apply(
        pd.to_numeric,
        errors="raise"
    ).to_numpy(dtype=float)

    if not np.isfinite(codes).all():
        raise ValueError("形状编号存在空值或无穷值")

    if not np.equal(codes, np.floor(codes)).all():
        raise ValueError("形状编号应为整数")

    return codes


train_shape_codes = get_shape_codes(df_train_valid)
test_shape_codes = get_shape_codes(df_test_valid)

X_train_with_shape = np.column_stack([
    X_train,
    train_shape_codes
])

X_test_with_shape = np.column_stack([
    X_test,
    test_shape_codes
])


def shape_change_features(shape_codes):
    before = shape_codes[:, 0]
    after = shape_codes[:, 1]

    valid_pair = (before > 0) & (after > 0)
    changed = valid_pair & (before != after)

    return np.column_stack([
        valid_pair.astype(float),
        changed.astype(float)
    ])


def make_shape_appearance_model():
    try:
        shape_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        shape_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    transformer = ColumnTransformer(
        transformers=[
            (
                "appearance",
                "passthrough",
                list(range(appearance_dimension))
            ),
            (
                "shape_categories",
                shape_encoder,
                [
                    appearance_dimension,
                    appearance_dimension + 1
                ]
            ),
            (
                "shape_change",
                FunctionTransformer(
                    shape_change_features,
                    validate=False
                ),
                [
                    appearance_dimension,
                    appearance_dimension + 1
                ]
            )
        ],
        sparse_threshold=0
    )

    return Pipeline([
        ("features", transformer),
        ("classifier", clone(classifier))
    ])


print("训练样本数：", len(X_train_with_shape))
print("测试样本数：", len(X_test_with_shape))

训练样本数： 280
测试样本数： 71


In [289]:
shape_oof_probabilities = np.full(
    (len(cv_y_train), 2),
    np.nan
)

for fold, (fit_index, valid_index) in enumerate(cv_folds, start=1):

    shape_fold_model = make_shape_appearance_model()

    shape_fold_model.fit(
        X_train_with_shape[fit_index],
        cv_y_train[fit_index]
    )

    fold_probabilities = shape_fold_model.predict_proba(
        X_train_with_shape[valid_index]
    )

    fold_classes = shape_fold_model.named_steps[
        "classifier"
    ].classes_

    for column, label in enumerate(fold_classes):
        shape_oof_probabilities[valid_index, int(label)] = (
            fold_probabilities[:, column]
        )

    print(f"形状联合模型：第 {fold} 折完成")

assert np.isfinite(shape_oof_probabilities).all()

shape_cv_rows = []

for threshold in cv_candidates:
    shape_predictions = cv_predict_with_threshold(
        shape_oof_probabilities,
        threshold
    )

    shape_cv_rows.append({
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_train, shape_predictions)
    })

shape_cv_threshold_table = pd.DataFrame(shape_cv_rows)

shape_cv_eligible = shape_cv_threshold_table.loc[
    shape_cv_threshold_table["coverage"] >= CV_MIN_COVERAGE
].copy()

if shape_cv_eligible.empty:
    raise ValueError("形状联合模型没有阈值满足预设覆盖率要求")

shape_cv_ranked = shape_cv_eligible.sort_values(
    by=[
        "macro_f1",
        "fail_recall_all",
        "coverage",
        "confidence_threshold"
    ],
    ascending=[False, False, False, True]
)

shape_best_threshold = float(
    shape_cv_ranked.iloc[0]["confidence_threshold"]
)

print("形状联合模型选择的置信度阈值：", shape_best_threshold)
print("固定概率差阈值：", CV_GAP_THRESHOLD)

display(shape_cv_ranked.head(10).round(4))

形状联合模型：第 1 折完成
形状联合模型：第 2 折完成
形状联合模型：第 3 折完成
形状联合模型：第 4 折完成
形状联合模型：第 5 折完成
形状联合模型选择的置信度阈值： 0.5
固定概率差阈值： 0.2


,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
0,0.50,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
1,0.51,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
2,0.52,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
3,0.53,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
4,0.54,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
5,0.55,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
6,0.56,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
7,0.57,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
8,0.58,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9
9,0.59,0.9286,0.9594,0.9679,0.0321,0.8867,0.963,0.7027,0.963,0.973,0.7222,9


In [291]:
shape_final_model = make_shape_appearance_model()

shape_final_model.fit(
    X_train_with_shape,
    cv_y_train
)

shape_raw_test_probabilities = shape_final_model.predict_proba(
    X_test_with_shape
)

shape_test_probabilities = np.zeros((len(cv_y_test), 2))

shape_classes = shape_final_model.named_steps[
    "classifier"
].classes_

for column, label in enumerate(shape_classes):
    shape_test_probabilities[:, int(label)] = (
        shape_raw_test_probabilities[:, column]
    )

shape_test_predictions = cv_predict_with_threshold(
    shape_test_probabilities,
    shape_best_threshold
)

results_shape = pd.DataFrame({
    "ID": df_test_valid["ID"].to_numpy(),
    "true_label": cv_y_test,
    "prediction": shape_test_predictions,
    "prob_0": shape_test_probabilities[:, 0],
    "prob_1": shape_test_probabilities[:, 1],
    "top_probability": shape_test_probabilities.max(axis=1),
    "probability_gap": np.abs(
        shape_test_probabilities[:, 0]
        - shape_test_probabilities[:, 1]
    ),
    "before_shape": test_shape_codes[:, 0],
    "after_shape": test_shape_codes[:, 1]
})

results_shape["uncertain_reason"] = [
    "; ".join(
        reason
        for condition, reason in [
            (top < shape_best_threshold, "最高概率不足"),
            (gap < CV_GAP_THRESHOLD, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results_shape["top_probability"],
        results_shape["probability_gap"]
    )
]

table_shape = pd.crosstab(
    results_shape["true_label"],
    results_shape["prediction"]
).reindex(
    index=[0, 1],
    columns=[0, 1, 2],
    fill_value=0
)

table_shape.index = ["真实 Pass", "真实 Fail"]
table_shape.columns = ["预测 Pass", "预测 Fail", "Uncertain"]

print("MobileNet＋形状联合模型的测试混淆矩阵")
display(table_shape)

shape_comparison_rows = []

comparison_settings = [
    ("原模型：手动 0.70", cv_test_probabilities, 0.70),
    ("原模型：手动 0.65", cv_test_probabilities, 0.65),
    ("原模型：手动 0.60", cv_test_probabilities, 0.60),
    (
        "原模型：交叉验证",
        cv_test_probabilities,
        cv_best_threshold
    ),
    (
        "MobileNet＋颜色：交叉验证",
        color_test_probabilities,
        color_best_threshold
    ),
    (
        "MobileNet＋形状：交叉验证",
        shape_test_probabilities,
        shape_best_threshold
    )
]

for name, probs, threshold in comparison_settings:
    comparison_predictions = cv_predict_with_threshold(
        probs,
        threshold
    )

    shape_comparison_rows.append({
        "method": name,
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_test, comparison_predictions)
    })

shape_test_comparison = pd.DataFrame(shape_comparison_rows)

display(shape_test_comparison.round(4))

print("真实 Fail 的预测明细")
display(
    results_shape.loc[
        results_shape["true_label"] == 1
    ].sort_values(["prediction", "prob_1"])
)

MobileNet＋形状联合模型的测试混淆矩阵


,预测 Pass,预测 Fail,Uncertain
真实 Pass,60,1,0
真实 Fail,3,7,0


,method,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
0,原模型：手动 0.70,0.70,0.6338,0.8491,0.7465,0.2535,0.5278,0.7049,0.2,0.4000,0.7,0.2857,18
1,原模型：手动 0.65,0.65,0.6901,0.8448,0.8169,0.1831,0.5456,0.7705,0.2,0.4000,0.8,0.2500,13
2,原模型：手动 0.60,0.60,0.7465,0.8413,0.8873,0.1127,0.5572,0.8361,0.2,0.3333,0.8,0.2500,8
3,原模型：交叉验证,0.50,0.7465,0.8413,0.8873,0.1127,0.5572,0.8361,0.2,0.3333,0.8,0.2500,8
4,MobileNet＋颜色：交叉验证,0.61,0.7887,0.8485,0.9296,0.0704,0.6592,0.8525,0.4,0.5000,1.0,0.4000,5
5,MobileNet＋形状：交叉验证,0.50,0.9437,0.9437,1.0000,0.0000,0.8728,0.9836,0.7,0.8750,1.0,0.7000,0


真实 Fail 的预测明细


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,before_shape,after_shape,uncertain_reason
12,C0301,1,0,0.976313,0.023687,0.976313,0.952626,76.0,76.0,
44,C0274,1,0,0.961550,0.038450,0.961550,0.923101,18.0,18.0,
35,C0273,1,0,0.790658,0.209342,0.790658,0.581316,18.0,18.0,
52,C0181,1,1,0.167868,0.832132,0.832132,0.664264,33.0,20.0,
57,C0385,1,1,0.106525,0.893475,0.893475,0.786949,44.0,68.0,
43,C0272,1,1,0.075978,0.924022,0.924022,0.848043,18.0,79.0,
61,C0361,1,1,0.065351,0.934649,0.934649,0.869299,7.0,18.0,
45,C0020,1,1,0.050158,0.949842,0.949842,0.899683,42.0,14.0,
56,C0087,1,1,0.034615,0.965385,0.965385,0.930770,78.0,54.0,
53,C0493,1,1,0.021053,0.978947,0.978947,0.957893,33.0,58.0,


# color + shape

In [294]:
# 列顺序：
# MobileNet 特征、before_color、after_color、
# before_shape、after_shape

X_train_with_attributes = np.column_stack([
    X_train,
    train_color_codes,
    train_shape_codes
])

X_test_with_attributes = np.column_stack([
    X_test,
    test_color_codes,
    test_shape_codes
])


def attribute_change_features(codes):
    before_color = codes[:, 0]
    after_color = codes[:, 1]
    before_shape = codes[:, 2]
    after_shape = codes[:, 3]

    color_valid = (before_color > 0) & (after_color > 0)
    shape_valid = (before_shape > 0) & (after_shape > 0)

    color_changed = color_valid & (before_color != after_color)
    shape_changed = shape_valid & (before_shape != after_shape)

    both_valid = color_valid & shape_valid

    # 两类信息都有效时，才计算联合变化
    either_changed = both_valid & (
        color_changed | shape_changed
    )

    both_changed = both_valid & (
        color_changed & shape_changed
    )

    return np.column_stack([
        color_valid,
        shape_valid,
        color_changed,
        shape_changed,
        both_valid,
        either_changed,
        both_changed
    ]).astype(float)


def make_color_shape_appearance_model():
    try:
        attribute_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        attribute_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    attribute_indices = [
        appearance_dimension,
        appearance_dimension + 1,
        appearance_dimension + 2,
        appearance_dimension + 3
    ]

    transformer = ColumnTransformer(
        transformers=[
            (
                "appearance",
                "passthrough",
                list(range(appearance_dimension))
            ),
            (
                "attribute_categories",
                attribute_encoder,
                attribute_indices
            ),
            (
                "attribute_changes",
                FunctionTransformer(
                    attribute_change_features,
                    validate=False
                ),
                attribute_indices
            )
        ],
        sparse_threshold=0
    )

    return Pipeline([
        ("features", transformer),
        ("classifier", clone(classifier))
    ])


assert len(X_train_with_attributes) == len(cv_y_train)
assert len(X_test_with_attributes) == len(cv_y_test)

print("训练样本数：", len(X_train_with_attributes))
print("测试样本数：", len(X_test_with_attributes))

训练样本数： 280
测试样本数： 71


In [296]:
joint_oof_probabilities = np.full(
    (len(cv_y_train), 2),
    np.nan
)

for fold, (fit_index, valid_index) in enumerate(cv_folds, start=1):

    joint_fold_model = make_color_shape_appearance_model()

    joint_fold_model.fit(
        X_train_with_attributes[fit_index],
        cv_y_train[fit_index]
    )

    fold_probabilities = joint_fold_model.predict_proba(
        X_train_with_attributes[valid_index]
    )

    fold_classes = joint_fold_model.named_steps[
        "classifier"
    ].classes_

    for column, label in enumerate(fold_classes):
        joint_oof_probabilities[valid_index, int(label)] = (
            fold_probabilities[:, column]
        )

    print(f"颜色＋形状联合模型：第 {fold} 折完成")

assert np.isfinite(joint_oof_probabilities).all()

joint_cv_rows = []

for threshold in cv_candidates:
    joint_predictions = cv_predict_with_threshold(
        joint_oof_probabilities,
        threshold
    )

    joint_cv_rows.append({
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_train, joint_predictions)
    })

joint_cv_threshold_table = pd.DataFrame(joint_cv_rows)

joint_cv_eligible = joint_cv_threshold_table.loc[
    joint_cv_threshold_table["coverage"] >= CV_MIN_COVERAGE
].copy()

if joint_cv_eligible.empty:
    raise ValueError("联合模型没有阈值满足预设覆盖率要求")

joint_cv_ranked = joint_cv_eligible.sort_values(
    by=[
        "macro_f1",
        "fail_recall_all",
        "coverage",
        "confidence_threshold"
    ],
    ascending=[False, False, False, True]
)

joint_best_threshold = float(
    joint_cv_ranked.iloc[0]["confidence_threshold"]
)

print("联合模型选择的置信度阈值：", joint_best_threshold)
print("固定概率差阈值：", CV_GAP_THRESHOLD)

display(joint_cv_ranked.head(10).round(4))

颜色＋形状联合模型：第 1 折完成
颜色＋形状联合模型：第 2 折完成
颜色＋形状联合模型：第 3 折完成
颜色＋形状联合模型：第 4 折完成
颜色＋形状联合模型：第 5 折完成
联合模型选择的置信度阈值： 0.62
固定概率差阈值： 0.2


,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
12,0.62,0.9607,0.9818,0.9786,0.0214,0.9312,0.9877,0.7838,1.0,0.9189,0.8529,6
0,0.50,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
1,0.51,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
2,0.52,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
3,0.53,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
4,0.54,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
5,0.55,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
6,0.56,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
7,0.57,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5
8,0.58,0.9607,0.9782,0.9821,0.0179,0.9302,0.9877,0.7838,1.0,0.9459,0.8286,5


In [298]:
joint_final_model = make_color_shape_appearance_model()

joint_final_model.fit(
    X_train_with_attributes,
    cv_y_train
)

joint_raw_test_probabilities = joint_final_model.predict_proba(
    X_test_with_attributes
)

joint_test_probabilities = np.zeros((len(cv_y_test), 2))

joint_classes = joint_final_model.named_steps[
    "classifier"
].classes_

for column, label in enumerate(joint_classes):
    joint_test_probabilities[:, int(label)] = (
        joint_raw_test_probabilities[:, column]
    )

joint_test_predictions = cv_predict_with_threshold(
    joint_test_probabilities,
    joint_best_threshold
)

results_joint = pd.DataFrame({
    "ID": df_test_valid["ID"].to_numpy(),
    "true_label": cv_y_test,
    "prediction": joint_test_predictions,
    "prob_0": joint_test_probabilities[:, 0],
    "prob_1": joint_test_probabilities[:, 1],
    "top_probability": joint_test_probabilities.max(axis=1),
    "probability_gap": np.abs(
        joint_test_probabilities[:, 0]
        - joint_test_probabilities[:, 1]
    ),
    "before_color": test_color_codes[:, 0],
    "after_color": test_color_codes[:, 1],
    "before_shape": test_shape_codes[:, 0],
    "after_shape": test_shape_codes[:, 1]
})

results_joint["uncertain_reason"] = [
    "; ".join(
        reason
        for condition, reason in [
            (top < joint_best_threshold, "最高概率不足"),
            (gap < CV_GAP_THRESHOLD, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results_joint["top_probability"],
        results_joint["probability_gap"]
    )
]

table_joint = pd.crosstab(
    results_joint["true_label"],
    results_joint["prediction"]
).reindex(
    index=[0, 1],
    columns=[0, 1, 2],
    fill_value=0
)

table_joint.index = ["真实 Pass", "真实 Fail"]
table_joint.columns = ["预测 Pass", "预测 Fail", "Uncertain"]

print("MobileNet＋颜色＋形状的测试混淆矩阵")
display(table_joint)

all_model_settings = [
    ("原模型：手动 0.70", cv_test_probabilities, 0.70),
    ("原模型：手动 0.65", cv_test_probabilities, 0.65),
    ("原模型：手动 0.60", cv_test_probabilities, 0.60),
    (
        "原模型：交叉验证",
        cv_test_probabilities,
        cv_best_threshold
    ),
    (
        "图像＋颜色：交叉验证",
        color_test_probabilities,
        color_best_threshold
    ),
    (
        "图像＋形状：交叉验证",
        shape_test_probabilities,
        shape_best_threshold
    ),
    (
        "图像＋颜色＋形状：交叉验证",
        joint_test_probabilities,
        joint_best_threshold
    )
]

all_model_rows = []

for name, probs, threshold in all_model_settings:
    model_predictions = cv_predict_with_threshold(
        probs,
        threshold
    )

    all_model_rows.append({
        "method": name,
        "confidence_threshold": threshold,
        **cv_evaluate(cv_y_test, model_predictions)
    })

all_model_comparison = pd.DataFrame(all_model_rows)

display(all_model_comparison.round(4))

print("真实 Fail 的预测明细")
display(
    results_joint.loc[
        results_joint["true_label"] == 1
    ].sort_values(["prediction", "prob_1"])
)

print("全部误判样本，不包含拒判")
display(
    results_joint.loc[
        (results_joint["prediction"] != 2)
        & (
            results_joint["prediction"]
            != results_joint["true_label"]
        )
    ]
)

MobileNet＋颜色＋形状的测试混淆矩阵


,预测 Pass,预测 Fail,Uncertain
真实 Pass,59,1,1
真实 Fail,1,8,1


,method,confidence_threshold,accuracy_all,accuracy_decided,coverage,abstention_rate,macro_f1,pass_recall,fail_recall_all,fail_precision,fail_coverage,fail_recall_decided,uncertain_count
0,原模型：手动 0.70,0.70,0.6338,0.8491,0.7465,0.2535,0.5278,0.7049,0.2,0.4000,0.7,0.2857,18
1,原模型：手动 0.65,0.65,0.6901,0.8448,0.8169,0.1831,0.5456,0.7705,0.2,0.4000,0.8,0.2500,13
2,原模型：手动 0.60,0.60,0.7465,0.8413,0.8873,0.1127,0.5572,0.8361,0.2,0.3333,0.8,0.2500,8
3,原模型：交叉验证,0.50,0.7465,0.8413,0.8873,0.1127,0.5572,0.8361,0.2,0.3333,0.8,0.2500,8
4,图像＋颜色：交叉验证,0.61,0.7887,0.8485,0.9296,0.0704,0.6592,0.8525,0.4,0.5000,1.0,0.4000,5
5,图像＋形状：交叉验证,0.50,0.9437,0.9437,1.0000,0.0000,0.8728,0.9836,0.7,0.8750,1.0,0.7000,0
6,图像＋颜色＋形状：交叉验证,0.62,0.9437,0.9710,0.9718,0.0282,0.9087,0.9672,0.8,0.8889,0.9,0.8889,2


真实 Fail 的预测明细


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,before_color,after_color,before_shape,after_shape,uncertain_reason
12,C0301,1,0,0.721386,0.278614,0.721386,0.442773,50.0,31.0,76.0,76.0,
52,C0181,1,1,0.134244,0.865756,0.865756,0.731512,52.0,52.0,33.0,20.0,
35,C0273,1,1,0.129400,0.870600,0.870600,0.741200,83.0,29.0,18.0,18.0,
45,C0020,1,1,0.078473,0.921527,0.921527,0.843054,3.0,3.0,42.0,14.0,
61,C0361,1,1,0.058779,0.941221,0.941221,0.882441,63.0,63.0,7.0,18.0,
43,C0272,1,1,0.048872,0.951128,0.951128,0.902256,83.0,83.0,18.0,79.0,
56,C0087,1,1,0.029702,0.970298,0.970298,0.940596,25.0,25.0,78.0,54.0,
57,C0385,1,1,0.018032,0.981968,0.981968,0.963936,86.0,86.0,44.0,68.0,
53,C0493,1,1,0.012228,0.987772,0.987772,0.975543,92.0,98.0,33.0,58.0,
44,C0274,1,2,0.428409,0.571591,0.571591,0.143182,33.0,74.0,18.0,18.0,最高概率不足; 前两类概率接近


全部误判样本，不包含拒判


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,before_color,after_color,before_shape,after_shape,uncertain_reason
12,C0301,1,0,0.721386,0.278614,0.721386,0.442773,50.0,31.0,76.0,76.0,
21,C0044,0,1,0.042328,0.957672,0.957672,0.915343,1.0,1.0,82.0,1.0,


# Human-Uncertain

In [301]:
from pathlib import Path

human_uncertain_excel = Path(
    "../data/labeled_data/2_Uncertain/"
    "Human-Uncertain/Human-Uncertain.xlsx"
).resolve()

labeled_root = Path("../data/labeled_data").resolve()

df_human_uncertain = pd.read_excel(human_uncertain_excel)

image_columns = [
    "before_1", "before_2", "before_3",
    "after_1", "after_2", "after_3"
]

# 使用副本转换路径，原 Excel 保持不变
df_human_ready = df_human_uncertain.copy()

# 原表路径例如：
# Human-Uncertain/img/C0011/before_1.png
#
# 转换为现有 build_event_feature 所需的：
# /2_Uncertain/Human-Uncertain/img/C0011/before_1.png
for column in image_columns:
    converted_paths = []

    for value in df_human_ready[column]:
        if pd.isna(value) or not str(value).strip():
            converted_paths.append(np.nan)
            continue

        image_path = (
            human_uncertain_excel.parent.parent
            / str(value).strip()
        ).resolve()

        converted_paths.append(
            "/" + image_path.relative_to(labeled_root).as_posix()
        )

    df_human_ready[column] = converted_paths


# 单独记录跳过的样本，避免全部失败时 np.stack 报错
human_features = []
human_valid_positions = []
human_skipped = []

for position, (_, row) in enumerate(df_human_ready.iterrows()):
    try:
        feature = build_event_feature(row).numpy()

        if not np.isfinite(feature).all():
            raise ValueError("图像特征包含 NaN 或 Inf")

        human_features.append(feature)
        human_valid_positions.append(position)

    except Exception as error:
        human_skipped.append({
            "ID": row["ID"],
            "error": str(error)
        })
        print(f"跳过 video {row['ID']}: {error}")

if not human_features:
    raise ValueError("没有成功提取特征的 Human-Uncertain 样本")

X_human_uncertain = np.stack(human_features)

df_human_valid = df_human_ready.iloc[
    human_valid_positions
].reset_index(drop=True).copy()

human_skipped_table = pd.DataFrame(
    human_skipped,
    columns=["ID", "error"]
)

print("Human-Uncertain 总数：", len(df_human_uncertain))
print("成功提取特征：", len(df_human_valid))
print("跳过数量：", len(human_skipped_table))

if not human_skipped_table.empty:
    display(human_skipped_table)

Human-Uncertain 总数： 7
成功提取特征： 7
跳过数量： 0


In [303]:
human_color_codes = get_color_codes(df_human_valid)
human_shape_codes = get_shape_codes(df_human_valid)

# 与联合模型训练时保持相同的列顺序
X_human_with_attributes = np.column_stack([
    X_human_uncertain,
    human_color_codes,
    human_shape_codes
])

assert (
    X_human_with_attributes.shape[1]
    == X_train_with_attributes.shape[1]
)

# 只预测，不调用 fit
human_raw_probabilities = joint_final_model.predict_proba(
    X_human_with_attributes
)

human_probabilities = np.zeros((len(df_human_valid), 2))

human_model_classes = joint_final_model.named_steps[
    "classifier"
].classes_

for column, label in enumerate(human_model_classes):
    human_probabilities[:, int(label)] = (
        human_raw_probabilities[:, column]
    )

human_predictions = cv_predict_with_threshold(
    human_probabilities,
    joint_best_threshold
)

results_human_uncertain = pd.DataFrame({
    "ID": df_human_valid["ID"].to_numpy(),

    # 单独记录人工类别，不作为二分类准确率的真实标签
    "human_category": "Human-Uncertain",

    "prediction": human_predictions,
    "prob_0": human_probabilities[:, 0],
    "prob_1": human_probabilities[:, 1],
    "top_probability": human_probabilities.max(axis=1),
    "probability_gap": np.abs(
        human_probabilities[:, 0] - human_probabilities[:, 1]
    ),
    "before_color": human_color_codes[:, 0],
    "after_color": human_color_codes[:, 1],
    "before_shape": human_shape_codes[:, 0],
    "after_shape": human_shape_codes[:, 1]
})

results_human_uncertain["prediction_name"] = (
    results_human_uncertain["prediction"].map({
        0: "Pass",
        1: "Fail",
        2: "Abstain"
    })
)

results_human_uncertain["abstention_reason"] = [
    "; ".join(
        reason
        for condition, reason in [
            (top < joint_best_threshold, "最高概率不足"),
            (gap < CV_GAP_THRESHOLD, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results_human_uncertain["top_probability"],
        results_human_uncertain["probability_gap"]
    )
]

print("使用的置信度阈值：", joint_best_threshold)
print("使用的概率差阈值：", CV_GAP_THRESHOLD)

display(results_human_uncertain)

使用的置信度阈值： 0.62
使用的概率差阈值： 0.2


,ID,human_category,prediction,prob_0,prob_1,top_probability,probability_gap,before_color,after_color,before_shape,after_shape,prediction_name,abstention_reason
0,C0011,Human-Uncertain,1,0.109910,0.890090,0.890090,0.780180,27.0,27.0,42.0,26.0,Fail,
1,C0295,Human-Uncertain,1,0.239248,0.760752,0.760752,0.521503,62.0,62.0,76.0,23.0,Fail,
2,C0296,Human-Uncertain,1,0.182017,0.817983,0.817983,0.635965,62.0,62.0,76.0,23.0,Fail,
3,C0299,Human-Uncertain,1,0.229159,0.770841,0.770841,0.541682,50.0,50.0,76.0,23.0,Fail,
4,C0308,Human-Uncertain,1,0.183205,0.816795,0.816795,0.633589,14.0,14.0,76.0,23.0,Fail,
5,C0509,Human-Uncertain,1,0.074979,0.925021,0.925021,0.850042,76.0,76.0,16.0,25.0,Fail,
6,C0579,Human-Uncertain,1,0.069384,0.930616,0.930616,0.861231,15.0,15.0,74.0,83.0,Fail,


In [305]:
human_prediction_counts = (
    results_human_uncertain["prediction"]
    .value_counts()
    .reindex([0, 1, 2], fill_value=0)
)

human_prediction_summary = pd.DataFrame({
    "模型输出": ["Pass", "Fail", "Abstain"],
    "数量": human_prediction_counts.to_numpy()
})

human_prediction_summary["占成功预测样本比例"] = (
    human_prediction_summary["数量"]
    / len(results_human_uncertain)
)

display(
    human_prediction_summary.style.format({
        "占成功预测样本比例": "{:.2%}"
    })
)

human_coverage = (
    results_human_uncertain["prediction"] != 2
).mean()

print(f"覆盖率：{human_coverage:.2%}")
print(f"拒判率：{1 - human_coverage:.2%}")
print("这些样本没有确定的 Pass/Fail 真值，不计算分类准确率。")

,模型输出,数量,占成功预测样本比例
0,Pass,0,0.00%
1,Fail,7,100.00%
2,Abstain,0,0.00%


覆盖率：100.00%
拒判率：0.00%
这些样本没有确定的 Pass/Fail 真值，不计算分类准确率。
